## Wine Quality Prediction using Artificial Neural Networks (ANN)

- Run a hyperparameter sweep on a training script

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform


In [31]:
import numpy as np
import pandas as pd 
import keras
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models import infer_signature
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from mlflow.models import validate_serving_input, convert_input_example_to_serving_input

## Load the Dataset

In [5]:
df=pd.read_csv("https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",sep=";",)

In [6]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


## Split the data into training,validation and test sets

In [7]:
df_train,df_test=train_test_split(df,test_size=0.20,random_state=42)

In [9]:
df_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4665,7.3,0.17,0.36,8.20,0.028,44.0,111.0,0.99272,3.14,0.41,12.4,6
1943,6.3,0.25,0.44,11.60,0.041,48.0,195.0,0.99680,3.18,0.52,9.5,5
3399,5.6,0.32,0.33,7.40,0.037,25.0,95.0,0.99268,3.25,0.49,11.1,6
843,6.9,0.19,0.35,1.70,0.036,33.0,101.0,0.99315,3.21,0.54,10.8,7
2580,7.7,0.30,0.26,18.95,0.053,36.0,174.0,0.99976,3.20,0.50,10.4,5


In [13]:
df_train[['quality']].values.ravel()

array([6, 5, 6, ..., 6, 6, 8], shape=(3918,))

In [14]:
train_x=df_train.drop(['quality'],axis=1).values
train_y=df_train[['quality']].values.ravel()
## test dataset
test_x=df_test.drop(['quality'],axis=1).values
test_y=df_test[['quality']].values.ravel()
## splitting this train data into train and validation
train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)
signature=infer_signature(train_x,train_y)

In [15]:
np.mean(train_x,axis=0)

array([6.87086790e+00, 2.77890874e-01, 3.33854499e-01, 6.36080089e+00,
       4.55105297e-02, 3.49082642e+01, 1.37556637e+02, 9.94029683e-01,
       3.19041800e+00, 4.89722399e-01, 1.05203340e+01])

In [16]:
train_x.shape[1]

11

## Creating the Artificial Neural Network model (ANN model)

## Builidng,training the model

In [39]:
def train_and_log_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):

    # =========================
    # MODEL
    # =========================
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    model = keras.Sequential([
        keras.Input(shape=(train_x.shape[1],)),
        keras.layers.Normalization(mean=mean, variance=var),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],
            momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # =========================
    # MLFLOW RUN
    # =========================
    with mlflow.start_run() as run:

        # -------------------------
        # TRAIN
        # -------------------------
        history = model.fit(
            train_x, train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
            verbose=0
        )

        # -------------------------
        # EVALUATION
        # -------------------------
        train_rmse = model.evaluate(train_x, train_y, verbose=0)[1]
        val_rmse = model.evaluate(valid_x, valid_y, verbose=0)[1]
        test_rmse = model.evaluate(test_x, test_y, verbose=0)[1]

        # -------------------------
        # LOG PARAMETERS
        # -------------------------
        mlflow.log_params(params)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", 64)

        # -------------------------
        # LOG METRICS
        # -------------------------
        mlflow.log_metric("train_rmse", train_rmse)
        mlflow.log_metric("val_rmse", val_rmse)
        mlflow.log_metric("test_rmse", test_rmse)

        # log per-epoch loss
        for i, loss in enumerate(history.history["loss"]):
            mlflow.log_metric("train_loss", loss, step=i)

        for i, val_loss in enumerate(history.history["val_loss"]):
            mlflow.log_metric("val_loss", val_loss, step=i)

        # -------------------------
        # PREDICTIONS (ARTIFACT)
        # -------------------------
        preds = model.predict(test_x)

        pred_df = pd.DataFrame({
            "actual": test_y.flatten(),
            "predicted": preds.flatten()
        })

        pred_df.to_csv("predictions.csv", index=False)
        mlflow.log_artifact("predictions.csv")

        # -------------------------
        # SIGNATURE + MODEL LOGGING
        # -------------------------
        input_example = train_x[:5]
        signature = infer_signature(input_example, model.predict(input_example))

        mlflow.tensorflow.log_model(
            model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )

        run_id = run.info.run_id

    return run_id

In [40]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Wine_Quality_Project")

params = {"lr": 0.01, "momentum": 0.9}

run_id = train_and_log_model(
    params,
    epochs=20,
    train_x=train_x,
    train_y=train_y,
    valid_x=valid_x,
    valid_y=valid_y,
    test_x=test_x,
    test_y=test_y
)

print("Run ID:", run_id)

Traceback (most recent call last):
  File "/workspaces/codespaces-blank/Wine Quallity Prediction/venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/codespaces-blank/Wine Quallity Prediction/venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/codespaces-blank/Wine Quallity Prediction/venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/codespaces-blank/Wine Quallity Prediction/venv/lib/python3.12

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


2026/03/26 17:03:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Run ID: eb468f5e0b3e441e906f8baa5bc6a43d


In [41]:
from mlflow.models import validate_serving_input, convert_input_example_to_serving_input

model_uri = f"runs:/{run_id}/model"

serving_payload = convert_input_example_to_serving_input(test_x[:5])

validate_serving_input(model_uri, serving_payload)

print("✅ Model validated successfully!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
✅ Model validated successfully!
